# Publication-Gate Pipeline Run

This notebook wraps `examples/run_publication_gate_pipeline.py`, which launches the user-run heavy gates: locked dynamic-A validation, empirical null controls, empirical re-estimation stability, readiness reporting, evidence packaging, and todo auditing.

By default it prints the pipeline commands without executing them. Set `RUN_PIPELINE = True` when you are ready.

<!-- reviewer-resume-contract -->
## Execution and resume contract

This notebook is aligned with the reviewer-revision implementation. Expensive work is checkpointed and safe to restart with the same configuration. Do not change methods, seeds, thresholds, or output paths while resuming. Saved outputs remain provisional until the compute-machine run and verification gates complete.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "calcium_transient_rising_flank").is_dir():
            return candidate
        nested = candidate / "calcium-transient-rising-flank"
        if (nested / "src" / "calcium_transient_rising_flank").is_dir():
            return nested
    raise RuntimeError("Run from the repository, package root, or notebooks directory.")


PROJECT_ROOT = find_project_root()
PYTHON = sys.executable
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
def command_text(command: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def run_or_print(command: list[str], *, execute: bool) -> None:
    print(command_text(command))
    if not execute:
        print("Dry run only. Set RUN_PIPELINE = True to execute.")
        return
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT / "src")
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
    env.setdefault("XDG_CACHE_HOME", "/tmp/font-cache")
    subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)

In [ ]:
RUN_PIPELINE = False

METHODS = "cgc,cgc-star"
EVENT_MODES = "compressed,physical"
DYNAMIC_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "validation_results" / "dynamic_episodic_locked"
NULL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "empirical_null_controls"
STABILITY_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "empirical_stability"
DYNAMIC_N_SEEDS = 20
DYNAMIC_N_STEPS = 1500
DYNAMIC_N_SURROGATES = 1000
DYNAMIC_MAX_LAG = 1
DYNAMIC_TAU = None  # set e.g. 2 to test only lag 2
DYNAMIC_N_PASTS = None  # set e.g. 5 for five conditioning-history samples
DYNAMIC_RISE_WAVEFORM_LENGTH = 20
DYNAMIC_TOPOLOGY_MODE = "sequence"  # "sequence" or "generated"
FALL_STATE_MODE = "stochastic_independent"
FALL_INITIAL_CEILING_FRACTION = 1.0
EDGE_DROPOUT_PROBABILITY = 0.20
EDGE_ADDITION_PROBABILITY = 0.02
SOURCE_DROPOUT_PROBABILITY = 0.15
SOURCE_RECRUITMENT_PROBABILITY = 0.25
SOURCE_RECRUITMENT_EDGE_PROBABILITY = 0.25
NULL_REPLICATES = 20
STABILITY_BOOTSTRAP = 20
MIN_RISE_RUN_SAMPLES = 1
RISE_CANDIDATE_FILTER = False
RISE_MATCH_MIN_LAG = DYNAMIC_TAU if DYNAMIC_TAU is not None else 1
RISE_MATCH_MAX_LAG = DYNAMIC_TAU if DYNAMIC_TAU is not None else None
RISE_MATCH_MIN_OVERLAP_SAMPLES = None
RISE_MATCH_MIN_OVERLAP_FRACTION = 0.5
RISE_RUN_CONTEXT_SAMPLES = None  # None uses DYNAMIC_N_PASTS, or max lag when unset


SKIP_COMPLETED = True
SKIP_DYNAMIC = False
SKIP_NULL = False
SKIP_STABILITY = False


In [ ]:
command = [
    PYTHON,
    "examples/run_publication_gate_pipeline.py",
    "--resume",
    "--python",
    PYTHON,
    "--methods",
    METHODS,
    "--event-modes",
    EVENT_MODES,
    "--dynamic-output-dir",
    str(DYNAMIC_OUTPUT_DIR),
    "--null-output-dir",
    str(NULL_OUTPUT_DIR),
    "--stability-output-dir",
    str(STABILITY_OUTPUT_DIR),
    "--dynamic-n-seeds",
    str(DYNAMIC_N_SEEDS),
    "--dynamic-n-steps",
    str(DYNAMIC_N_STEPS),
    "--dynamic-n-surrogates",
    str(DYNAMIC_N_SURROGATES),
    "--dynamic-max-lag",
    str(DYNAMIC_MAX_LAG),
    "--dynamic-rise-waveform-length",
    str(DYNAMIC_RISE_WAVEFORM_LENGTH),
    "--dynamic-topology-mode",
    DYNAMIC_TOPOLOGY_MODE,
    "--fall-state-mode",
    FALL_STATE_MODE,
    "--fall-initial-ceiling-fraction",
    str(FALL_INITIAL_CEILING_FRACTION),
    "--edge-dropout-probability",
    str(EDGE_DROPOUT_PROBABILITY),
    "--edge-addition-probability",
    str(EDGE_ADDITION_PROBABILITY),
    "--source-dropout-probability",
    str(SOURCE_DROPOUT_PROBABILITY),
    "--source-recruitment-probability",
    str(SOURCE_RECRUITMENT_PROBABILITY),
    "--source-recruitment-edge-probability",
    str(SOURCE_RECRUITMENT_EDGE_PROBABILITY),
    "--min-rise-run-samples",
    str(MIN_RISE_RUN_SAMPLES),
    "--rise-match-min-lag",
    str(RISE_MATCH_MIN_LAG),
    "--rise-match-min-overlap-fraction",
    str(RISE_MATCH_MIN_OVERLAP_FRACTION),
    "--null-replicates",
    str(NULL_REPLICATES),
    "--stability-bootstrap",
    str(STABILITY_BOOTSTRAP),
]
if RISE_CANDIDATE_FILTER:
    command.append("--rise-candidate-filter")
if DYNAMIC_TAU is not None:
    command.extend(["--dynamic-tau", str(DYNAMIC_TAU)])
if DYNAMIC_N_PASTS is not None:
    command.extend(["--dynamic-n-pasts", str(DYNAMIC_N_PASTS)])
if RISE_MATCH_MAX_LAG is not None:
    command.extend(["--rise-match-max-lag", str(RISE_MATCH_MAX_LAG)])
if RISE_MATCH_MIN_OVERLAP_SAMPLES is not None:
    command.extend(["--rise-match-min-overlap-samples", str(RISE_MATCH_MIN_OVERLAP_SAMPLES)])
if RISE_RUN_CONTEXT_SAMPLES is not None:
    command.extend(["--rise-run-context-samples", str(RISE_RUN_CONTEXT_SAMPLES)])
if RUN_PIPELINE:
    command.append("--execute")
if SKIP_COMPLETED:
    command.append("--skip-completed")
if SKIP_DYNAMIC:
    command.append("--skip-dynamic")
if SKIP_NULL:
    command.append("--skip-null")
if SKIP_STABILITY:
    command.append("--skip-stability")

print("Publication gate configuration:")
print(f"  RUN_PIPELINE={RUN_PIPELINE}")
print(f"  SKIP_COMPLETED={SKIP_COMPLETED}")
print(f"  SKIP_DYNAMIC={SKIP_DYNAMIC}")
print(f"  SKIP_NULL={SKIP_NULL}")
print(f"  SKIP_STABILITY={SKIP_STABILITY}")
print(f"  DYNAMIC_OUTPUT_DIR={DYNAMIC_OUTPUT_DIR}")
print(f"  NULL_OUTPUT_DIR={NULL_OUTPUT_DIR}")
print(f"  STABILITY_OUTPUT_DIR={STABILITY_OUTPUT_DIR}")

run_or_print(command, execute=RUN_PIPELINE)


In [ ]:
status_files = {
    "dynamic": DYNAMIC_OUTPUT_DIR / "summary.json",
    "null_controls": NULL_OUTPUT_DIR / "summary.json",
    "stability": STABILITY_OUTPUT_DIR / "summary.json",
    "readiness": PROJECT_ROOT / "outputs" / "result_readiness" / "summary.json",
    "evidence": PROJECT_ROOT / "outputs" / "manuscript_evidence" / "summary.json",
    "todo_audit": PROJECT_ROOT / "outputs" / "todo_completion" / "summary.json",
}
for name, path in status_files.items():
    marker = "available" if path.is_file() else "missing"
    print(f"{name}: {marker} - {path}")